In [1]:
import os
import nltk
import pandas as pd
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv(), override=True)

HF_TOKEN  = os.getenv('HF_TOKEN')

In [ ]:
DATASETS_DIR = './sample_data'
DOWNLOAD_FILE_NAME = 'consumer-reviews-of-amazon-products'
DOWNLOAD_FILE_EXTENSION = 'zip'
DOWNLOAD_FILE = '.'.join([DOWNLOAD_FILE_NAME, DOWNLOAD_FILE_EXTENSION])
EXTRACT_FILES_DIR = os.path.join(DATASETS_DIR, DOWNLOAD_FILE_NAME)
DOWNLOAD_FILE_DIR = os.path.join(DATASETS_DIR, DOWNLOAD_FILE)

In [5]:
!curl -L -o $DOWNLOAD_FILE_DIR\
  https://www.kaggle.com/api/v1/datasets/download/datafiniti/consumer-reviews-of-amazon-products

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
  0      0   0      0   0      0      0      0                              0
  0      0   0      0   0      0      0      0                              0

  0      0   0      0   0      0      0      0                              0
 48 16.25M  48  7.87M   0      0  5.71M      0   00:02   00:01   00:01  7.88M
100 16.25M 100 16.25M   0      0 10.41M      0   00:01   00:01          7.88M
100 16.25M 100 16.25M   0      0 10.41M      0   00:01   00:01          7.88M
100 16.25M 100 16.25M   0      0 10.41M      0   00:01   00:01          7.88M


In [ ]:
import zipfile

with zipfile.ZipFile(DOWNLOAD_FILE_DIR, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_FILES_DIR)
    
os.remove(DOWNLOAD_FILE_DIR)

In [ ]:
FILES_DIR = [f for f in os.listdir(EXTRACT_FILES_DIR)]

frames = []
for filePath in FILES_DIR:
    data = pd.read_csv(os.path.join(EXTRACT_FILES_DIR, filePath), low_memory=False)
    data["source_file"] = filePath
    frames.append(data)
    print(f"{filePath:<62} {data.shape[0]:>6,} rows")

df = pd.concat(frames, ignore_index=True)
print(f"\n{'CONCATENATED':<62} {df.shape[0]:>6,} rows x {df.shape[1]} cols")
df.head(3)

1429_1.csv                                                     34,660 rows
Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv       5,000 rows
Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv 28,332 rows

CONCATENATED                                                   67,992 rows x 28 cols


,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,...,reviews.userCity,reviews.userProvince,reviews.username,source_file,dateAdded,dateUpdated,primaryCategories,imageURLs,manufacturerNumber,sourceURLs
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,NaN,NaN,Adapter,1429_1.csv,NaN,NaN,NaN,NaN,NaN,NaN
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,NaN,NaN,truman,1429_1.csv,NaN,NaN,NaN,NaN,NaN,NaN
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,NaN,NaN,DaveZ,1429_1.csv,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Check unique values
print(df['primaryCategories'].unique())
print(df['brand'].unique())

# Get columns containing relevant information
df_2 = df[['name', 'brand', 'asins', 'categories', 'manufacturer', 'reviews.date', 'reviews.username', 'primaryCategories', 'reviews.rating', 'reviews.text', 'reviews.title', 'reviews.dateSeen']]

# Remove duplicate reviews on dataFrame
df_2.drop_duplicates(subset=['name', 'reviews.username', 'reviews.title', 'reviews.text'], inplace=True)

# Get number of null values by columns
print(df_2.isna().sum())

<ArrowStringArray>
[                          nan,                 'Electronics',
        'Electronics,Hardware', 'Office Supplies,Electronics',
           'Electronics,Media',             'Health & Beauty',
             'Office Supplies',      'Animals & Pet Supplies',
               'Home & Garden',       'Electronics,Furniture',
    'Toys & Games,Electronics']
Length: 11, dtype: str
<ArrowStringArray>
[                      'Amazon',                  'Amazon Fire',
                  'Amazon Echo',                'Amazon Coco T',
               'Amazon Fire Tv', 'Amazon Digital Services Inc.',
                 'Amazonbasics',                 'AmazonBasics']
Length: 8, dtype: str
name                  6760
brand                    0
asins                    2
categories               0
manufacturer             0
reviews.date            39
reviews.username        11
primaryCategories    34660
reviews.rating          33
reviews.text             1
reviews.title           17
reviews.dateS

In [ ]:
from lib.data_preprocessing import data_frame_preprocessing

# Drop products without a 'name' and a 'review.text' because it is a vital field
df_2.dropna(subset=['reviews.text'], inplace=True)

# Fill with unique ASIN if missing name
df_2['name'] = df_2['name'].fillna(df_2['asins'])
df_2.drop(columns=['asins'], inplace=True)

# Fill missing categories and usernames with 'unknown' because this is critical data
df_2['primaryCategories'] = df_2['primaryCategories'].fillna('unknown')
df_2['reviews.username'] = df_2['reviews.username'].fillna('unknown')

# Fill missing titles to 'no title' because capturing the review text is more critical
df_2['reviews.title'] = df_2['reviews.title'].fillna('no title')

# Fill missing ratings with averange rating data
df_2['reviews.rating'] = df_2['reviews.rating'].fillna(df_2['reviews.rating'].mean())

# Join amazonbasics and amazonBasics brand as a typo in the same brand
df_2['brand'] = df_2['brand'].apply(lambda brand_name: 'Amazon Basics' if brand_name.lower().strip() == 'amazonbasics' else brand_name)

# Fill missing review dates with the last date seen to avoid NaN values, as this is the assumed review date. Then drop date seen column.
df_2['reviews.date'] = df_2['reviews.date'].fillna(df_2['reviews.dateSeen'])
df_2.drop(columns=['reviews.dateSeen'], inplace=True)

# Clean punctuation and stopwords from text categories
df_2 = data_frame_preprocessing(df_2, ['name', 'reviews.title', 'reviews.text'])

print(df_2['brand'].unique())
print(df_2.isna().sum())

# Get number of null values by columns
display(df_2.sample(10))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\agcor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\agcor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\agcor\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


<ArrowStringArray>
[                      'Amazon',                  'Amazon Fire',
                  'Amazon Echo',                'Amazon Coco T',
               'Amazon Fire Tv', 'Amazon Digital Services Inc.',
                'Amazon Basics']
Length: 7, dtype: str
name                 0
brand                0
categories           0
manufacturer         0
reviews.date         0
reviews.username     0
primaryCategories    0
reviews.rating       0
reviews.text         0
reviews.title        0
dtype: int64


,name,brand,categories,manufacturer,reviews.date,reviews.username,primaryCategories,reviews.rating,reviews.text,reviews.title
42387,AmazonBasics AAA Performance Alkaline Batterie...,Amazon Basics,"AA,AAA,Health,Electronics,Health & Household,C...",AmazonBasics,2016-07-16T00:00:00.000Z,ByLeslie Vesper,Health & Beauty,5.0,Great,Five Stars
58016,All-New Fire HD 8 Tablet 8 HD Display Wi-Fi 32...,Amazon,"Fire Tablets,Tablets,All Tablets,Amazon Tablet...",Amazon,2017-01-26T00:00:00.000Z,Turtle1,Electronics,5.0,Got 8 year old granddaughter loves Uses game p...,Great younger kids
36431,Fire Kids Edition Tablet 7 Display Wi-Fi 16 GB...,Amazon,"Computers,Fire Tablets,Electronics Features,Co...",Amazon,2017-05-06T00:00:00.000Z,Chas,Electronics,5.0,Bought 2 year old loves 's already got things ...,far good
60912,Fire Tablet Alexa 7 Display 16 GB Magenta Spec...,Amazon,"Fire Tablets,Android Tablets,Tablets,All Table...",Amazon,2017-05-12T00:00:00.000Z,SamJ,Electronics,5.0,bought niece loves Works great,Great kids great price
28159,B00L9EPT8O B01E6AO69U,Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock...",Amazon,2016-09-13T00:00:00.000Z,Mort1,unknown,5.0,past used Google information needed ask Alexa ...,works advertised
52422,Amazon Tap Smart Assistant Alexaenabled black ...,Amazon,"Amazon Echo,Home Theater & Audio,MP3 MP4 Playe...",Amazon,2016-09-09T00:00:00.000Z,DPC3,Electronics,5.0,purchased Tap thought portable speaker echo 's...,Tap
49602,AmazonBasics AA Performance Alkaline Batteries...,Amazon Basics,"AA,AAA,Electronics Features,Health,Electronics...",AmazonBasics,2015-11-29T00:00:00.000Z,ByC,Health & Beauty,5.0,batteries work expected noticeable difference mAh,Five Stars
52535,Kindle Voyage E-reader 6 High-Resolution Displ...,Amazon,"eBook Readers,Electronics Features,Walmart for...",Amazon,2015-05-15T00:00:00.000Z,SillyGoober,Electronics,5.0,Nothing replace joy flipping pages book living...,great space saver book lovers
19274,Amazon Kindle Paperwhite eBook reader 4 GB 6 m...,Amazon,"Walmart for Business,Office Electronics,Tablet...",Amazon,2017-03-15T00:00:00.000Z,Exs3,unknown,5.0,Great product Great value Worth every penny Ad...,Great product
34908,Amazon Echo Show Alexa-enabled Bluetooth Speak...,Amazon,"Computers,Amazon Echo,Virtual Assistant Speake...",Amazon,2017-12-01T00:00:00.000Z,Michelle,"Electronics,Hardware",5.0,Love echo show much Easy use connects mobil ph...,Echo show easy use


In [ ]:
from lib.data_preprocessing import rating_to_sentiment

# Map 'reviews.rating' to labels category ['positive', 'neutral', 'negative']
def rating_to_sentiment(r):
    if r <= 2:
        return "negative"
    if r == 3:
        return "neutral"
    return "positive"

df_2['sentiment'] = df_2['reviews.rating'].apply(rating_to_sentiment)

display(df_2)

,name,brand,categories,manufacturer,reviews.date,reviews.username,primaryCategories,reviews.rating,reviews.text,reviews.title,sentiment
0,All-New Fire HD 8 Tablet 8 HD Display Wi-Fi 16...,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",Amazon,2017-01-13T00:00:00.000Z,Adapter,unknown,5.0,product far disappointed children love use lik...,Kindle,positive
1,All-New Fire HD 8 Tablet 8 HD Display Wi-Fi 16...,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",Amazon,2017-01-13T00:00:00.000Z,truman,unknown,5.0,great beginner experienced person Bought gift ...,fast,positive
2,All-New Fire HD 8 Tablet 8 HD Display Wi-Fi 16...,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",Amazon,2017-01-13T00:00:00.000Z,DaveZ,unknown,5.0,Inexpensive tablet use learn step NABI thrille...,Beginner tablet 9 year old son,positive
3,All-New Fire HD 8 Tablet 8 HD Display Wi-Fi 16...,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",Amazon,2017-01-13T00:00:00.000Z,Shacks,unknown,4.0,'ve Fire HD 8 two weeks love tablet great valu...,Good,positive
4,All-New Fire HD 8 Tablet 8 HD Display Wi-Fi 16...,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",Amazon,2017-01-12T00:00:00.000Z,explore42,unknown,5.0,bought grand daughter comes visit set user ent...,Fantastic Tablet kids,positive
...,...,...,...,...,...,...,...,...,...,...,...
67987,Fire HD 8 Tablet Alexa 8 HD Display 16 GB Tang...,Amazon,"Fire Tablets,Tablets,All Tablets,Amazon Tablet...",Amazon,2016-12-07T00:00:00.000Z,Mom2twinsplus1,Electronics,5.0,got 2 8 yr old twins 11 yr old one one better ...,Xmas gift,positive
67988,Fire HD 8 Tablet Alexa 8 HD Display 16 GB Tang...,Amazon,"Fire Tablets,Tablets,All Tablets,Amazon Tablet...",Amazon,2017-01-20T00:00:00.000Z,fireman21,Electronics,4.0,bought niece Christmas gift.she 9 years old love,yes great tablet,positive
67989,Fire HD 8 Tablet Alexa 8 HD Display 16 GB Tang...,Amazon,"Fire Tablets,Tablets,All Tablets,Amazon Tablet...",Amazon,2017-01-30T00:00:00.000Z,suzannalicious,Electronics,5.0,nice light internet browsing keeping top email...,get lot price,positive
67990,Fire HD 8 Tablet Alexa 8 HD Display 16 GB Tang...,Amazon,"Fire Tablets,Tablets,All Tablets,Amazon Tablet...",Amazon,2017-02-17T00:00:00.000Z,SandyJ,Electronics,5.0,Tablet absolutely everything want watch TV Sho...,get entire World less 100,positive


In [11]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df_2, test_size=0.20, random_state=42, stratify=df_2["sentiment"])
train_df, val_df = train_test_split(train_df, test_size=0.10, random_state=42, stratify=train_df["sentiment"])

In [ ]:
import torch

c:\Users\agcor\Develop\ironhack_ai\7_Project3\project-automated-customers-reviews\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_ID = 'cardiffnlp/twitter-roberta-base-sentiment-latest'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

In [ ]:
from datasets import Dataset

LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL = {value: key for key, value in LABEL2ID.items()}

# Convert sentiments label2id
def dataset_label2id(df, label):
    return df.map(lambda row: { 'labels': LABEL2ID[row[label]] })

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128)

# Convert data frame to data set for hugging face model
def df_to_dataset(df):
    ds = Dataset.from_pandas(df[['reviews.text', 'sentiment']].rename(columns={'reviews.text': 'text'}))
    ds = dataset_label2id(ds, 'sentiment')
    return ds.map(tokenize_fn, batched=True)

train_ds = df_to_dataset(train_df)
test_ds = df_to_dataset(test_df)
val_ds = df_to_dataset(val_df)

Map: 100%|██████████| 5227/5227 [00:00<00:00, 36574.25 examples/s]


In [22]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13634.83it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "key", "value"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 1,035,267 || all params: 125,683,206 || trainable%: 0.8237


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

working_dir = './'
output_directory = os.path.join(working_dir, "peft_lab_outputs")

training_args = TrainingArguments(
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-5,
    num_train_epochs=2,
    use_cpu=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

c:\Users\agcor\Develop\ironhack_ai\7_Project3\project-automated-customers-reviews\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
peft_model_path = os.path.join(output_directory, f"lora_model")
trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)

In [ ]:
from peft import PeftModel

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
)
fine_tuned = PeftModel.from_pretrained(base_model, peft_model_path)

In [ ]:
def predict(texts):
    texts = [texts] if isinstance(texts, str) else list(texts)
    enc = tokenizer(texts, truncation=True, max_length=128, padding=True, return_tensors="pt")
    with torch.no_grad():
        probs = fine_tuned(**enc).logits.softmax(-1)
    preds = probs.argmax(-1)
    return [(ID2LABEL[i.item()], p[i].item()) for p, i in zip(probs, preds)]

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

y_pred = trainer.predict(test_ds).predictions.argmax(1)
y_true = test_df["label"].values

print(classification_report(y_true, y_pred, target_names=LABEL2ID, digits=3))
print(f"macro-F1: {f1_score(y_true, y_pred, average='macro'):.3f}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt=".2f", cmap="Blues",
            xticklabels=LABEL2ID, yticklabels=LABEL2ID)
plt.xlabel("predicted"); plt.ylabel("actual"); plt.title("Confusion matrix (row-normalised)")
plt.show()